# Running inference with Multicellular Transformer

This notebook outlines how to run inference with a pretrained Multicellular Transformer model. 

The embeddings of the Multicellular Transformer model are highly contextual and can be used to:
* identify co-regulated genes by computing cosine similarity between pairs of genes or clustering genes into gene sets
* discover condition specific genes by interrogating the neighborhood of genes across conditions
* find enriched receptor-ligand pairs

This tutorial describes how to run inference and get contextual gene embeddings using different data types as inputs, namely:
* Cell Arrays object (https://github.com/BiocPy/cellarr)
* anndata object (https://github.com/scverse/anndata) with scRNA-seq data only

## Step 1 - setup Datamodule

### Cell Arrays
Cell Arrays https://github.com/BiocPy/cellarr is a python package which provides a convenient way to store and access large collections of genomics data, such as scRNA-seq by using Tile-DB backed store. Here, we use it for storing our pretraining corpus consisting ~14M cells across various datasets.

In [2]:
import os
import json

from cell_cell_interactions.pp.cellarr_datamodule import CellarrDataModule
from lightning import seed_everything

seed_everything(32)

# specify the input dir, for Cell Arrays this is the input directory with the cellarr dataset
cellarr_input_dir = "/data/bucket/wiatrakm/cell-cell-interactions/input-data/cellarr-data/corpus-14M-cells/"

# load cell type mapping path to harmonise cell types across datasets
cell_type_mapping_file_path = os.path.join(cellarr_input_dir, "cell_type_map.json")
with open(cell_type_mapping_file_path) as f:
    cell_type_mapping = json.load(f)
            
train_split_prop = 0.95
batch_size = 4
datamodule = CellarrDataModule(
    cell_tdb_uri=os.path.join(cellarr_input_dir, "cell_metadata"),
    gene_tdb_uri=os.path.join(cellarr_input_dir, "gene_annotation"),
    counts_tdb_uri=os.path.join(cellarr_input_dir, "counts"),
    val_samples=None,
    val_set_proportion=1 - train_split_prop,
    label_column="cell_type",
    study_column="study",
    sample_column="sample",
    tissue_column="tissue",
    batch_size=batch_size,
    sample_size=1024,
    num_workers=1,
    min_sample_size=1024,
    min_cell_type_size=0.001,
    agg_cell_type_gex=False,
    sample_sampling="weighted",
    cell_sampling="stratified",
    pad_value_expression=0.,
    mask_prob=0.,
    mask_tissue_prob=0.,
    cell_type_sqrt_sampling=False,
    perc_min_expressed_cells_per_gene=0.01,
    max_seq_len=12000,
    lognorm=True,
    target_sum=1e4,
    random_seed=32,
    cell_type_mapping=cell_type_mapping,
)

val_dataloader = datamodule.val_dataloader()

[rank: 0] Seed set to 32


Training data size: (13119364, 12)
Validation data size: (697726, 12)


### anndata scRNA-seq only

Multicellular Transformer also supports reading directly anndata objects. Here we show how to use it to read lung cancer atlas. The model will automatically adapt to the available cell type proportions.

To ensure consistency about genes used during pretraining and inference, we read the `gene2idx` and  dict established during pretraining.

In [11]:
from cell_cell_interactions.pp.adata_rnaseq_datamodule import AdataRnaseqDataModule


adata_file_path = "/data/bucket/wiatrakm/cell-cell-interactions/input-data/lung-cancer-atlas/lung_cancer_atlas_processed_all_genes_log1pnorm.h5ad"
gene2idx_file_path = "/data/bucket/wiatrakm/cell-cell-interactions/output-data/runs-lung-cancer-finetuning/lung_cancer_lr00015_alpha0.01_agg/gene2idx.json"


with open(gene2idx_file_path) as f:
    gene2idx = json.load(f)
idx2gene = {idx: gene for gene, idx in gene2idx.items()}


# get the data
datamodule = AdataRnaseqDataModule(
    adata_file_path=adata_file_path,
    gene2idx=gene2idx,
    val_set_proportion=1 - 0.8,
    sample_size=512,
    sample_column="sample",
    label_column="cell_type_major",
    tissue_column="tissue",
    min_sample_size=512,
    min_cell_type_size=0.002,
    sample_sampling="weighted",
    cell_sampling="stratified",
    batch_size=1,
    num_workers=4,
    agg_cell_type_gex=True,
    pad_value_expression=0.,
    mask_prob=0.,
    mask_tissue_prob=0.,
    perc_min_expressed_cells_per_gene=0.01,
    max_seq_len=12000,
    random_seed=32,
    backed=True,  # do not load anndata object into memory, but in backed mode
)

Training data size: (1026803, 53)
Validation data size: (231608, 53)


## Step 2 - load pretrained model

In [ ]:
import torch

from cell_cell_interactions.modeling.model import MulticellularTransformerPLModel


ckpt_path = "/Users/maciejwiatrak/Downloads/mct_checkpoint_12k.ckpt"

# load model from checkpoint and move it to a GPU
model = MulticellularTransformerPLModel.load_from_checkpoint(ckpt_path).cuda()
# set the model to the eval mode
model.eval()

MulticellularTransformerPLModel(
  (model): MulticellularTransformerModule(
    (encoder): MulticellularTransformerEncoder(
      (gene_id_emb): Embedding(58056, 512, padding_idx=0)
      (tissue_emb): Embedding(93, 512, padding_idx=0)
      (gene_expression_encoder): Sequential(
        (0): Linear(in_features=1024, out_features=768, bias=True)
        (1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (2): ReLU()
        (3): Dropout(p=0.2, inplace=False)
        (4): Linear(in_features=768, out_features=512, bias=True)
      )
      (gene_expression_mask_emb): Embedding(1, 512)
      (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.2, inplace=False)
      (cat_cov_emb): ModuleDict()
      (encoder): TransformerEncoder(
        (layers): ModuleList(
          (0-11): 12 x TransformerEncoderLayer(
            (self_attn): MultiheadAttention(
              (out_proj): NonDynamicallyQuantizableLinear(in_features=512, out_fe

## Step 3 - run inference

Run model inference using a pretrained model.

In [4]:
# fetch a batch
batch = next(iter(val_dataloader))

/homefs/home/wiatrakm/miniforge3/envs/ccc/lib/python3.10/site-packages/tiledb/ctx.py:560: UserWarning: TileDB is a multithreading library and deadlocks are likely if fork() is called after a TileDB context has been created (such as for array access). To safely use TileDB with multiprocessing or concurrent.futures, choose 'spawn' as the start method for child processes. For example: multiprocessing.set_start_method('spawn').
  warnings.warn(


In [5]:
# run inference
with torch.no_grad():
    output = model.model(
        expression_matrix=batch.expression_matrix.to(model.device),
        cat_covariates=batch.cat_covariates.to(model.device),
        gene_ids=batch.gene_ids.to(model.device),
        is_masked_gene_expression=batch.is_masked_gene_expression.to(model.device),
        padding_mask=batch.padding_mask.to(model.device),
        tissue=batch.tissue_idx.to(model.device),
    )

# fetch last hidden state
last_hidden_state = output.last_hidden_state

/homefs/home/wiatrakm/miniforge3/envs/ccc/lib/python3.10/site-packages/torch/nn/modules/transformer.py:408: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. (Triggered internally at ../aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(output, src_key_padding_mask.logical_not(), mask_check=False)


## Extension - subset CellArray to specific dataset and/or condition

It is also possible to subset data to study and/or conditions of interest, such as healthy samples in Lupus dataset available in the corpus. Below is an example of this.

In [7]:
datamodule = CellarrDataModule(
    cell_tdb_uri=os.path.join(cellarr_input_dir, "cell_metadata"),
    gene_tdb_uri=os.path.join(cellarr_input_dir, "gene_annotation"),
    counts_tdb_uri=os.path.join(cellarr_input_dir, "counts"),
    val_samples=None,
    val_set_proportion=1 - train_split_prop,
    label_column="cell_type",
    study_column="study",
    sample_column="sample",
    tissue_column="tissue",
    batch_size=batch_size,
    sample_size=1024,
    num_workers=1,
    min_sample_size=1024,
    min_cell_type_size=0.001,
    agg_cell_type_gex=False,
    sample_sampling="weighted",
    cell_sampling="stratified",
    pad_value_expression=0.,
    mask_prob=0.,
    mask_tissue_prob=0.,
    cell_type_sqrt_sampling=False,
    perc_min_expressed_cells_per_gene=0.01,
    max_seq_len=12000,
    lognorm=True,
    target_sum=1e4,
    random_seed=32,
    cell_type_mapping=cell_type_mapping,
    # restrict to conditions accepts a dict as input, where key is the column name and value is the desired value
    restrict_to_conditions={
        'study': '/scratch/site/u/wiatrakm/cell-cell-interactions/data/input-data/datasets/lupus/lupus_preprocessed_all_genes.h5ad',
        'disease': 'normal',
    }
)

val_dataloader = datamodule.val_dataloader()

Training data size: (463960, 12)
Validation data size: (20144, 12)


To look into what studies and conditions are available in the pretraining corpus, use the `datamodule.data_df` DataFrame which shows the metadata for every cell and sample.

In [8]:
datamodule.data_df.head()

,sample,tissue,assay,cell_type,cell_type_ontology_term_id,disease,sex,cell_type_remapped_v2,study,cellarr_sample,cellarr_cell_index_in_sample,study::::sample,idx
6712074,0a1148dc-356f-4dc4-9c32-bf0823257d93,blood,10x 3' v2,CD4-positive T cells,CL:0000624,normal,female,Helper T Cells,/scratch/site/u/wiatrakm/cell-cell-interaction...,sample_5,0,/scratch/site/u/wiatrakm/cell-cell-interaction...,6712074
6712081,98e974a9-c08e-4066-8930-d627b18f1a89,blood,10x 3' v2,CD4-positive T cells,CL:0000624,normal,female,Helper T Cells,/scratch/site/u/wiatrakm/cell-cell-interaction...,sample_5,7,/scratch/site/u/wiatrakm/cell-cell-interaction...,6712081
6712083,a51ffc5a-e4cb-4b47-9154-4514e1dc9d65,blood,10x 3' v2,B cells,CL:0000236,normal,female,B cell,/scratch/site/u/wiatrakm/cell-cell-interaction...,sample_5,9,/scratch/site/u/wiatrakm/cell-cell-interaction...,6712083
6712084,bc16e1fe-cd25-4a6b-a329-2e2c7121ad8d,blood,10x 3' v2,CD4-positive T cells,CL:0000624,normal,female,Helper T Cells,/scratch/site/u/wiatrakm/cell-cell-interaction...,sample_5,10,/scratch/site/u/wiatrakm/cell-cell-interaction...,6712084
6712085,869da0bb-4487-493b-86ca-e601eee46196,blood,10x 3' v2,CD4-positive T cells,CL:0000624,normal,female,Helper T Cells,/scratch/site/u/wiatrakm/cell-cell-interaction...,sample_5,11,/scratch/site/u/wiatrakm/cell-cell-interaction...,6712085


## Extension - get zero-shot pretrained embeddings

In [ ]:
ckpt_path = "/Users/maciejwiatrak/Downloads/mct_checkpoint_12k.ckpt"

# get zero-shot embeddings by fetching the weights of the classifer head
zs_embeddings = torch.load(ckpt_path, map_location="cpu")['state_dict']['model.gene_id_classifier.decoder.weight']
# get gene2idx map mapping index of embedding to Ensembl ID
gene2idx = datamodule.gene2idx